# Part 3: Principal Component Analysis

---

## Import Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from sklearn.decomposition import PCA
from scipy.stats import mannwhitneyu

## Prepare Data

In [ ]:
# Read in counts table.
ccle_counts_subset = pd.read_csv('outs/1_ccle_counts_subset.csv', header=0, index_col=0)
ccle_counts_subset.index.name = 'Gene'

# Read in meta table.
ccle_meta_subset = pd.read_csv('outs/1_ccle_meta_subset.csv', header=0, index_col=0)

# Remove lowly expressed genes in counts matrix
keep = (ccle_counts_subset.sum(axis=1) >=100)
ccle_counts_subset = ccle_counts_subset[keep].copy()

# Transpose cleaned counts matrix
ccle_counts_subset = ccle_counts_subset.transpose().astype(int)

# QC counts and meta tables
print(ccle_counts_subset.shape)
print(ccle_meta_subset.shape)

# Variance-stablize counts data using PyDeseq2
inference = DefaultInference(n_cpus=6)
dds = DeseqDataSet(
    counts=ccle_counts_subset,
    metadata=ccle_meta_subset,
    design="~Pathology",
    refit_cooks=True,
    inference=inference,
)
dds.deseq2()
dds.vst()
vst_df = pd.DataFrame(
    dds.layers['vst_counts'],
    index=dds.obs_names,
    columns=dds.var_names
)
print(vst_df)

## Principal Component Analysis (PCA) in RNA-seq

With variance-stabilized counts prepared, we can proceed to PCA—a cornerstone of Exploratory Data Analysis (EDA) for high-dimensional biological data.

### What is PCA?
PCA is a dimensionality reduction technique designed to capture the most informative sources of variation in a dataset. By condensing thousands of individual gene expression dimensions into a few **Principal Components (PCs)**, it simplifies complex transcriptomic patterns into a readable overview of sample relationships.

### The "Space" of Gene Expression
To visualize this, imagine each of our 176 samples as a single point plotted in a ~17,000-dimensional space, where every axis represents the expression level of one gene. Because many genes are co-regulated (highly correlated), this "cloud" of points actually follows specific directions of variation. PCA mathematically rotates this coordinate system, creating new axes—the PCs—that are linear combinations of these correlated genes. 
* **PC1** is aligned to capture the largest possible variance in the data.
* **PC2** captures the second largest, and so on.

### Diagnostic and Exploratory Value
PCA serves as a critical quality control gate. By mapping our samples onto the first two or three PCs, we can immediately identify the dominant forces driving our data:

* **Technical Confounders:** If PC1 aligns with sequencing dates, batch numbers, or preparation sites, it indicates that technical "noise" is overshadowing the biology. This signal warns that batch correction is required before proceeding.
* **Biological Signal:** If PC1 or PC2 cleanly separates samples by treatment status, tissue type, or disease subtype (e.g., SCLC vs. NSCLC), it confirms that the experimental variables are the primary drivers of transcriptomic change.

By reducing ~17,000 genes down to a handful of components, PCA allows us to verify the integrity of the experiment and intuitively explore the biological structure of the dataset.

### Implementation in Python
Unlike the R-based `DESeq2` package, **PyDeseq2** does not feature a built-in `plotPCA` function. Instead, it follows the Python philosophy of modularity, leveraging the specialized capabilities of **scikit-learn** for statistical computing and **Seaborn** or **Matplotlib** for visualization.

To implement this, we utilize the `PCA` class from the `sklearn.decomposition` module, which uses Singular Value Decomposition (SVD) alogorithm to caluclate PCs, eigenvalues, and loading matrix. Here are two key implementation details:

* **Data Orientation:** We ensure our input DataFrame is structured as **(Samples $\times$ Genes)**.
* **Feature Selection:** We will focus our analysis on the top highly variable genes. This reduces noise from constitutively expressed genes and highlights the features most likely to drive biological separation between our samples.

In [ ]:
# Select top 500 highly variable genes.
top_genes = vst_df.var(axis=0).sort_values(ascending=False).head(500).index
vst_df_top = vst_df[top_genes]
print(f"Top 500 variable counts shape: {vst_df_top.shape}")

# Instantiate pca for first four PCs.
pca = PCA(n_components=4)

# Get first two PCs for all samples.
components = pca.fit_transform(vst_df_top)
print(f"Shape of PC matrix: {components.shape}")

# Create a dataframe for plotting and add condition column.
pca_df = pd.DataFrame(
    data=components,
    columns=['PC1', 'PC2', 'PC3', 'PC4'],
    index=vst_df_top.index
)
pca_df = pd.concat([pca_df, ccle_meta_subset], axis=1)
pca_df

In [ ]:
# Extract explained variance ratios from pca object.
exp_var = pca.explained_variance_ratio_ * 100
condition = 'Pathology'

# Plot samoples on first two PCs.
fig, axes = plt.subplots(1, 2, figsize=(10, 4), squeeze=True)
ax1, ax2 = axes.flatten()

# Subplot1
sns.scatterplot(ax=ax1, data=pca_df, x='PC1', y='PC2', hue=condition, s=50, alpha=0.5)
ax1.set_xlabel(f"PC1 ({exp_var[0]:0.1f}% variance)")
ax1.set_ylabel(f"PC2 ({exp_var[1]:0.1f}% variance)")
ax1.set_title("PC1 vs PC2")
ax1.legend(loc='best')

# Subplot2
sns.scatterplot(ax=ax2, data=pca_df, x='PC3', y='PC4', hue=condition, s=50, alpha=0.5)
ax2.set_xlabel(f"PC3 ({exp_var[2]:0.1f}% variance)")
ax2.set_ylabel(f"PC4 ({exp_var[3]:0.1f}% variance)")
ax2.set_title("PC3 vs PC4")
ax2.legend(loc='best')
plt.show()

### Visualizing Variance: The Scree (Elbow) Plot

To determine the relative importance of each Principal Component, we can generate a **Scree Plot** (commonly referred to as an **"Elbow Plot"**. This visualization displays the percentage of total variance explained by each individual PC. In high-dimensional RNA-seq datasets, the first few components typically capture the majority of the biological signal, while subsequent components represent decreasing amounts of variation—often reaching a "floor" where only technical noise remains.

#### Identifying the "Elbow":
The "elbow" of the plot is the point where the curve flattens significantly. This serves as a heuristic to decide how many PCs are necessary to interpret. For instance, if the first three PCs capture 80% of the variance and the curve plateaus thereafter, focusing on those three provides a high-fidelity, simplified summary of your samples.

In [ ]:
pca_all = PCA()
components_all = pca_all.fit_transform(vst_df_top)
exp_var_all = pca_all.explained_variance_ratio_ * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4), squeeze=True)
ax1, ax2 = axes.flatten()
pcs = range(1, len(exp_var_all)+1)

ax1.plot(pcs, exp_var_all, 'o-')
ax1.set_xlabel('Principal Components')
ax1.set_ylabel('Percentage of Variance Explained')
ax1.set_title('Scree Plot')
ax1.set_xlim(1, 25)

ax2.plot(pcs, exp_var_all.cumsum(), 'o-')
ax2.set_xlabel('Principal Components')
ax2.set_ylabel('Cumulative Percentage of Variance Explained')
ax2.set_title('Cumulative Scree Plot')
ax2.set_xlim(1, 25)
ax2.set_ylim(0, None);

The "elbow" in the left plot above serves as a clear cutoff for dimensionality reduction. **PC1 & PC2** capture the vast majority of the dataset's variance. Notice the sharp "bend" at PC3, where the curve flattens significantly.

The plateau from PC3 onwards indicates that subsequent components likely represent stochastic noise or minor technical artifacts rather than meaningful biological structure. Focusing our analysis on **PC1 and PC2** provides the most robust, high-fidelity summary of the differences between our samples.

To see exactly how much of our total data signal is retained in these first two dimensions, we look at cumulative explained variancve on the right plot above. 

### Interpreting the PC1 vs. PC2 Projection

The PCA plot above reveals a clear, bimodal segregation of lung cancer cell lines, primarily driven by **PC1**. This horizontal separation suggests that the single largest source of variation in our dataset is also the factor dividing these samples into two distinct clusters.

#### 1. Analyzing the Segregation
The existence of a "left group" and a "right group" is not a guaranteed outcome of PCA; it occurs only when the underlying data contains a dominant, discrete feature. To determine the significance of this split, we must identify the nature of PC1:

* **Technical Noise:** If PC1 correlates with metadata like "extraction date" or "sequencing batch," the clusters are likely artifacts of **batch effects**. In this case, the separation is a hurdle to be corrected rather than a discovery.
* **Biological Signal:** If PC1 aligns with biological metadata—such as histological subtype (SCLC vs. NSCLC), a specific mutation, or drug sensitivity—then we have identified the primary driver of transcriptomic diversity in our study.

#### 2. Determining the Driver
By "overlaying" our metadata onto this plot (e.g., coloring points by cell line origin), we can visually confirm whether this PC1-driven variance represents a compelling biological breakthrough or a technical confounder requiring adjustment.

We can programmatically check if our metadata explains this separation by calculating the correlation between PC1 scores and our categorical variables:

In [ ]:
conditions = ['Pathology', 'Site_Primary', 'Gender', 'tcga_code']

fig, axes = plt.subplots(2, 2, figsize=(12, 8), squeeze=False)
axes = axes.flatten()
for i, condition in enumerate(conditions):
    ax = axes[i]
    sns.boxplot(ax=ax, data=pca_df, x=condition, y='PC1', color='skyblue', showfliers=False)
    ax.set_title(condition)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

The boxplot analysis confirms that PC1 captures a significant biological signal, effectively segregating the SCLC cohort from the LUAD/LUSC cell lines. This suggests that the primary axis of transcriptomic variation in this dataset is driven by histological subtype.

To back up this visual observation with a statistic, we can run a quick Mann-Whitney U test on the PC1 scores. We choose Mann-Whitney U test because SCLCC and LUAD/LUSC cohorts are skewed.

In [ ]:
# Group PC1 scores by subtypes
sclc_pc1 = pca_df[pca_df['tcga_code'] == 'SCLC']['PC1']
luad_lusc_pc1 = pca_df[pca_df['tcga_code'].isin(['LUAD', 'LUSC'])]['PC1']

# Compute p-value using Mann-Whitney U test
_, p_val = mannwhitneyu(sclc_pc1, luad_lusc_pc1)
print(f"P-value for PC1 separation: {p_val:.2e}")

### Analyzing Gene Loadings in PC1

In RNA-seq analysis, **gene loadings** represent the individual weight or contribution of each gene to a specific Principal Component. These values quantify how much an individual gene's expression profile influences the direction of a PC, serving as a mathematical bridge between abstract dimensions and biological reality.

#### Why Loadings Matter:
* **Biological Interpretation:** By identifying genes with the highest absolute loadings, we can assign functional meaning to a PC. For example, if PC1 is dominated by neuroendocrine markers, we can confidently interpret that axis as a "SCLC-ness" gradient.
* **Feature Importance:** Loadings reveal which genes are the primary drivers of the clusters observed in our score plot.
* **Directionality:** A positive loading indicates that higher gene expression pushes a sample to the right on the PC axis, while a negative loading pulls it to the left.

#### Quantitative Summary
| Term | Definition | Role in EDA |
| :--- | :--- | :--- |
| **PC Score** | Sample coordinate ($x, y$) | Identifies clusters and outliers. |
| **Gene Loading** | Weight of a gene in a PC | Identifies the biological "why" behind the clusters. |

By ranking these loadings, we transform a purely statistical transformation into a list of candidate biomarkers or pathways that define the differences between our lung cancer subtypes.

In [ ]:
# Create a dataframe for gene loadings in first four PCs
loadings_df = pd.DataFrame(
    data=pca.components_.T,
    columns=['PC1', 'PC2', 'PC3', 'PC4'],
    index=vst_df_top.columns.to_list()
)

# Get the top 10 genes pulling to the RIGHT
pc1_top_positives = loadings_df['PC1'].sort_values(ascending=False).head(10)

# Get the top 10 genes pulling to the LEFT
pc1_top_negatives = loadings_df['PC1'].sort_values(ascending=True).head(10)

# Get the top 10 genes by absolute IMPACT (could be all positive or mixed)
pc1_top_absolute = loadings_df['PC1'].abs().sort_values(ascending=False).head(10)

In [ ]:
pc1_top_positives

In [ ]:
pc1_top_negatives

In [ ]:
pc1_top_absolute

In [ ]:
top_positive_vst = vst_df_top[pc1_top_positives.index.to_list()]
top_positive_vst_long = top_positive_vst.melt(
    value_vars=pc1_top_positives.index.to_list(),
    var_name="Gene",
    value_name="Expression (VST)"    
)


top_negative_vst = vst_df_top[pc1_top_negatives.index.to_list()]
top_negative_vst_long = top_negative_vst.melt(
    value_vars=pc1_top_negatives.index.to_list(),
    var_name="Gene",
    value_name="Expression (VST)"    
)


top_absolute_vst = vst_df_top[pc1_top_absolute.index.to_list()]
top_absolute_vst_long = top_absolute_vst.melt(
    value_vars=pc1_top_absolute.index.to_list(),
    var_name="Gene",
    value_name="Expression (VST)"    
)

In [ ]:
top_genes_data = {
    'Positive': top_positive_vst_long,
    'Negative': top_negative_vst_long,
    'Absolute': top_absolute_vst_long
}

fig, axes = plt.subplots(3, 1, figsize=(8, 12), squeeze=False)
axes = axes.flatten()
for i, (type, df) in enumerate(top_genes_data.items()):
    ax=axes[i]
    sns.boxplot(ax=ax, data=df, x='Gene', y='Expression (VST)', showfliers=False, palette='tab10')
    sns.stripplot(ax=ax, data=df, x='Gene', y='Expression (VST)', jitter=True, alpha=0.5, s=2, color='black')
    ax.set_title(f"Top 10 {type} Genes")
    ax.set_xlabel("")

plt.tight_layout()
plt.show()

As illustrated, these ten genes exert the greatest influence on PC1, effectively defining the transcriptomic divergence between the observed cell line clusters.

Exploring our data through multiple lenses is a fundamental practice in bioinformatics, as each visualization highlights a different structural layer.

To bridge the gap between abstract PCA coordinates and actual gene expression, we will generate a faceted series of PC1-vs-PC2 plots. In this arrangement, each subplot represents one of our six candidate genes, with the sample points color-coded by their variance-stabilized counts.

This "feature plot" approach allows us to visually validate the PCA results: if a gene has a high loading on PC1, we should see a clear expression gradient moving from left to right across the plot. Now, let's see if we can visualize it.

In [ ]:
# Combine PCA dataframe with top genes 
top_positive_vst_combined = pd.concat([pca_df, top_positive_vst], axis=1)

# Unpivot dataframe to long dataframe
top_positive_vst_combined_long = top_positive_vst_combined.melt(
    id_vars=["PC1", "PC2"],
    value_vars=top_positive_vst.columns.to_list(),
    var_name="Gene",
    value_name="Expression (VST)"
)

# Plot PC1 vs PC2 for each of top ten driver genes

# Method 1
g = sns.relplot(
    data=top_positive_vst_combined_long,
    x="PC1", y="PC2",
    col="Gene", col_wrap=5,
    hue="Expression (VST)",
    # palette="coolwarm",
    palette=sns.blend_palette(["blue", "red"], as_cmap=True),
    kind="scatter",
    s=150
    )

# Method 2
# g = sns.FacetGrid(top_positive_vst_combined_long, col="Gene", col_wrap=5, height=2)
# g.map_dataframe(sns.scatterplot, x="PC1", y="PC2", hue="Expression (VST)", palette=sns.blend_palette(["blue", "red"], as_cmap=True))

# Set the size for x and y axis labels
g.set_axis_labels("PC1", "PC2", fontsize=16)

# Set the size for the titles of each facet (the "time" labels)
g.set_titles(col_template="{col_name}", size=16, weight='bold')

# Adjust the tick parameters (the numbers on the axes)
for ax in g.axes.flat:
    ax.tick_params(axis='both', labelsize=14)

# Adjust the font size of the labels (the category names)
plt.setp(g._legend.get_texts(), fontsize='14') 

# Adjust the font size of the legend title (e.g., "day")
plt.setp(g._legend.get_title(), fontsize='16')

As expected, the top ten genes with positive PC1 loadings drive the segregation of samples along the horizontal axis; samples with high expression of these features cluster toward the right, while those with lower expression are distributed toward the left.

In [ ]:
# Combine PCA dataframe with top genes 
top_negative_vst_combined = pd.concat([pca_df, top_negative_vst], axis=1)

# Unpivot dataframe to long dataframe
top_negative_vst_combined_long = top_negative_vst_combined.melt(
    id_vars=["PC1", "PC2"],
    value_vars=top_negative_vst.columns.to_list(),
    var_name="Gene",
    value_name="Expression (VST)"
)

# Plot PC1 vs PC2 for each of top ten driver genes
g = sns.relplot(
    data=top_negative_vst_combined_long,
    x="PC1", y="PC2",
    col="Gene", col_wrap=5,
    hue="Expression (VST)",
    # palette="coolwarm",
    palette=sns.blend_palette(["blue", "red"], as_cmap=True),
    kind="scatter",
    s=150
    )

# Set the size for x and y axis labels
g.set_axis_labels("PC1", "PC2", fontsize=16)

# Set the size for the titles of each facet (the "time" labels)
g.set_titles(col_template="{col_name}", size=16, weight='bold')

# Adjust the tick parameters (the numbers on the axes)
for ax in g.axes.flat:
    ax.tick_params(axis='both', labelsize=14)

# Adjust the font size of the labels (the category names)
plt.setp(g._legend.get_texts(), fontsize='14') 

# Adjust the font size of the legend title (e.g., "day")
plt.setp(g._legend.get_title(), fontsize='16');

Similary, the top ten genes with negative PC1 loadings drive the segregation of samples along the horizontal axis; samples with high expression of these features cluster toward the left, while those with lower expression are distributed toward the right.

### Molecular Identity and Lineage Drivers on PC1

The primary principal component (**PC1**) in this dataset serves as a mathematical proxy for cellular lineage. By analyzing the genes at the extreme ends of this axis, we can translate abstract statistical variance into a concrete understanding of lung cancer pathophysiology.

For broader context, see: [A Short Primer on Lung Cancer Biology](./primer_on_lung_cancer_biology.md).

#### 1\. The Positive Axis: Neuroendocrine Differentiation (SCLC)

Genes with the highest positive loadings define a **neuroendocrine (NE) signature**. In thoracic oncology, this profile is the definitive transcriptomic "divider" between Small Cell Lung Cancer (**SCLC**) and Non-Small Cell Lung Cancer (**NSCLC**).

**Top Positive PC1 Drivers:**

  * **Lineage-Defining Regulators:** *INSM1, ASCL1*
  * **Vesicle & Secretory Machinery:** *CHGA, SCG3, SYT4, SEZ6*
  * **Neuromimetic Factors:** *KIF1A, DPYSL5, BEX1, XKR7*

##### Functional Categorization

  * **Master Regulators (*ASCL1, INSM1*):** These transcription factors trigger the neuroendocrine program. *INSM1* is a high-specificity "gold standard" used in diagnostic pathology to identify SCLC.
  * **Secretory Machinery (*CHGA, SCG3, SYT4, SEZ6*):** These proteins facilitate the packaging and regulated release of neuropeptides. *SEZ6* is currently a primary therapeutic target for emerging antibody-drug conjugates (ADCs).
  * **Neural Co-option (*KIF1A, DPYSL5, BEX1, XKR7*):** Involved in axonal transport and neural development, these genes are hijacked by SCLC cells to drive their aggressive, neuron-like phenotype.

#### 2\. The Negative Axis: Epithelial-Mesenchymal Identity (NSCLC)

The negative end of PC1 represents an active **Epithelial-Mesenchymal (Non-Neuroendocrine)** program. These genes define the identity of **LUAD** and **LUSC**, focusing on cellular infrastructure, environmental sensing, and wound-healing pathways.

**Top Negative PC1 Drivers:**

  * **Epithelial Scaffolding:** *CAV1, MYOF, ANXA1*
  * **Matrix Remodeling:** *TGFBI, TGM2, S100A16*
  * **Signaling & Plasticity:** *FOSL1, OSMR, NT5E, GPRC5A*

##### Functional Categorization

  * **The Mesenchymal "Switch" (*FOSL1, TGFBI, TGM2*):** *FOSL1* (FRA-1) is a master transcription factor often driven by KRAS signaling. Along with *TGFBI*, it promotes an **Epithelial-to-Mesenchymal Transition (EMT)**, increasing cellular mobility and invasiveness.
  * **Membrane Architecture (*CAV1, MYOF, ANXA1*):** These genes build a robust plasma membrane scaffold. *Caveolin-1 (CAV1)* is a classic marker of the epithelial/mesenchymal state, providing the structural basis for complex signaling.
  * **Adaptive Signaling (*NT5E, OSMR, GPRC5A*):** These drivers manage the tumor's interaction with its microenvironment. *NT5E* (CD73) facilitates immune evasion, while *GPRC5A* is a lung-specific receptor often dysregulated in adenocarcinoma.

#### Summary: The Molecular Identity Toggle

PC1 effectively functions as a **Lineage Toggle** between two distinct biological states:

| Feature | Positive PC1 (SCLC) | Negative PC1 (NSCLC) |
| :--- | :--- | :--- |
| **Primary State** | **Neuroendocrine** (Synaptic-like) | **Epithelial/Mesenchymal** (Adhesion-rich) |
| **Master Driver** | *ASCL1* / *INSM1* | *FOSL1* / *TGFB* Pathway |
| **Cellular Goal** | Neuropeptide Secretion | Structural Integrity & Invasion |
| **Transcriptome** | "Neuron-mimetic" | "Mucosal-lining" |

<br>

> **The Takeaway:** A sample's position along PC1 is a quantitative measurement of its commitment to a **Neuroendocrine** lineage versus an **Epithelial/Mesenchymal** lineage. The negative end is not merely the "absence" of SCLC markers; it is a dedicated program of cellular scaffolding and environmental response.

Now let's do the same exact exercise for PC2 to understand what kind of biology it describes.